# Flower Classification: Fixing Overfitting with Data Augmentation

This notebook builds a small CNN to classify photos of flowers into 5 species (roses, daisy, dandelion, sunflowers, tulips), then demonstrates a very common real-world problem: a model that memorizes the training set instead of learning general patterns (**overfitting**), and one of the standard fixes for it - **data augmentation**, combined with **dropout**.

**Structure of this notebook:**
1. Load and explore the dataset
2. Build the image/label arrays
3. Train a baseline CNN and observe overfitting
4. Interpret the model's predictions
5. Apply data augmentation and dropout
6. Retrain and compare against the baseline

---

## 1. Setup

We import everything we'll need up front:

- **TensorFlow / Keras** - builds and trains the CNN.
- **OpenCV (`cv2`)** - reads and resizes images to a consistent size before we feed them to the model.
- **PIL** - used only for quickly previewing images in the notebook.
- **NumPy** - turns our lists of images/labels into arrays the model can consume.
- **Matplotlib** - for visualizing sample images and training curves.
- **scikit-learn** - for a clean train/test split.

We also fix a random seed, so re-running this notebook gives the same train/test split and the same "random" augmentations each time.

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import cv2
import PIL.Image
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print('TensorFlow version:', tf.__version__)

---

## 2. Loading the Dataset

We use TensorFlow's built-in flowers dataset - a public archive of ~3,700 photos across 5 flower categories, hosted by Google. `keras.utils.get_file` downloads and unpacks it once, then reuses the cached copy on any future run.

In [ ]:
dataset_url = ('https://storage.googleapis.com/download.tensorflow.org/'
               'example_images/flower_photos.tgz')

# untar=True automatically extracts the downloaded .tgz archive
data_dir = keras.utils.get_file(
    'flower_photos', origin=dataset_url, cache_dir='.', untar=True
)
data_dir = pathlib.Path(data_dir) / 'flower_photos'

image_count = len(list(data_dir.glob('*/*.jpg')))
print('Dataset location:', data_dir)
print('Total images found:', image_count)

### Exploring the folder structure

Each flower category lives in its own subfolder - this is a standard layout for image classification datasets, where the folder name doubles as the label. Let's see the categories and how many images each one has, so we know if the classes are roughly balanced.

In [ ]:
flowers_images_dict = {
    'roses': list(data_dir.glob('roses/*')),
    'daisy': list(data_dir.glob('daisy/*')),
    'dandelion': list(data_dir.glob('dandelion/*')),
    'sunflowers': list(data_dir.glob('sunflowers/*')),
    'tulips': list(data_dir.glob('tulips/*')),
}

# The label each category maps to -- these integers are what the
# model will actually learn to predict
flowers_labels_dict = {
    'roses': 0,
    'daisy': 1,
    'dandelion': 2,
    'sunflowers': 3,
    'tulips': 4,
}

for flower_name, images in flowers_images_dict.items():
    print(f'{flower_name:12s}: {len(images)} images')

A quick look at one sample photo from each category:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, (flower_name, images) in zip(axes, flowers_images_dict.items()):
    ax.imshow(PIL.Image.open(str(images[0])))
    ax.set_title(flower_name)
    ax.axis('off')
plt.tight_layout()
plt.show()

---

## 3. Building the Image and Label Arrays

Neural networks need every input image to be **the same size**. Our source photos come in many different resolutions, so we resize every one of them to a fixed `180x180` before anything else. We define that size once as a constant (`IMG_HEIGHT`, `IMG_WIDTH`) so it's used consistently everywhere in the notebook, instead of retyping `180` in multiple places - this matters later, since the augmentation layer needs to know this exact shape too.

For each image, we:
1. Read it with OpenCV (`cv2.imread`).
2. Resize it to `180x180`.
3. Append it to `X` (the inputs), and append its numeric label to `y` (the targets).

In [ ]:
IMG_HEIGHT = 180
IMG_WIDTH = 180

X, y = [], []

for flower_name, images in flowers_images_dict.items():
    for image_path in images:
        img = cv2.imread(str(image_path))
        resized_img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        X.append(resized_img)
        y.append(flowers_labels_dict[flower_name])

X = np.array(X)
y = np.array(y)

print('X shape:', X.shape, ' (num_images, height, width, channels)')
print('y shape:', y.shape)

---

## 4. Train/Test Split and Pixel Scaling

We hold out a portion of the data as a **test set** the model never trains on, so we can honestly measure how well it generalizes to new images.

We also **scale pixel values from `[0, 255]` down to `[0, 1]`**. Raw pixel values are just brightness intensities on an arbitrary 0-255 scale - neural networks train faster and more stably on small, normalized numbers than on large raw ones.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

X_train_scaled = X_train / 255.0
X_test_scaled = X_test / 255.0

print('Training images:', X_train_scaled.shape[0])
print('Test images:', X_test_scaled.shape[0])

---

## 5. Baseline CNN (No Augmentation)

We start with a straightforward CNN, with no protection against overfitting, so we can *see* the overfitting problem before we fix it.

### The architecture
- **`Conv2D(filters, 3, activation='relu')`** - a convolutional layer with a 3x3 sliding filter. Each `Conv2D` layer learns a set of small pattern detectors (edges, textures, colors) that slide across the image. We stack three of these (16 → 32 → 64 filters), so later layers can combine simple patterns from earlier layers into more complex ones.
- **`padding='same'`** - pads the image so the output has the same width/height as the input, instead of shrinking after every convolution.
- **`MaxPooling2D()`** - downsamples the feature map by taking the strongest activation in each small region, after each `Conv2D`. This reduces the amount of data flowing forward and makes the model less sensitive to the exact pixel position of a feature.
- **`Flatten()`** - converts the final 2D feature maps into a single flat list of numbers, so they can feed into `Dense` layers.
- **`Dense(128, activation='relu')`** - a fully-connected layer that combines all the extracted features together.
- **`Dense(num_classes)`** - the output layer, one number per flower category. Notice there's **no `softmax`** here - we leave these as raw scores ("logits") and instead tell the loss function to apply softmax internally via `from_logits=True`. This is a common, numerically more stable pattern in Keras.

### Training with a validation split
We pass `validation_split=0.2`, which carves out 20% of the *training* data purely to monitor performance on data the model isn't learning from during each epoch - this is what lets us actually see overfitting happening as training progresses (training accuracy climbing while validation accuracy lags behind or plateaus).

In [ ]:
num_classes = 5

baseline_model = Sequential([
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes),
])

baseline_model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

baseline_history = baseline_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=30,
    verbose=2,
)

### Visualizing the overfitting gap

If training accuracy keeps climbing toward 100% while validation accuracy stalls or falls behind, that gap **is** overfitting - the model is memorizing training images rather than learning generalizable flower features.

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['accuracy'], label='train accuracy')
    ax1.plot(history.history['val_accuracy'], label='val accuracy')
    ax1.set_title(f'{title} -- Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(history.history['loss'], label='train loss')
    ax2.plot(history.history['val_loss'], label='val loss')
    ax2.set_title(f'{title} -- Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.tight_layout()
    plt.show()


# NOTE: plot_history is a small, reusable plotting helper used
# twice below (baseline and augmented models) to keep the two
# comparable, rather than duplicating the same plotting code.
plot_history(baseline_history, 'Baseline Model')

In [ ]:
baseline_test_loss, baseline_test_accuracy = baseline_model.evaluate(
    X_test_scaled, y_test, verbose=0
)
print(f'Baseline test accuracy: {baseline_test_accuracy:.4f}')

---

## 6. Interpreting a Prediction

Because the model's final layer outputs raw logits (not probabilities), we apply `tf.nn.softmax` ourselves afterward to convert them into class probabilities that sum to 1. `np.argmax` then picks the class with the highest probability - the model's actual predicted label.

In [ ]:
predictions = baseline_model.predict(X_test_scaled, verbose=0)

# Look at the first test image as an example
probabilities = tf.nn.softmax(predictions[0])
predicted_label = np.argmax(probabilities)
true_label = y_test[0]

class_names = list(flowers_labels_dict.keys())
print('Predicted:', class_names[predicted_label],
      f'({100 * np.max(probabilities):.1f}% confidence)')
print('Actual:   ', class_names[true_label])

---

## 7. Data Augmentation

### Why this fixes overfitting
The baseline model above only ever sees each training photo **exactly as-is**, every epoch. With a few thousand images spread across 5 classes, it's easy for the model to start memorizing specific images instead of learning what actually makes a rose look like a rose.

Data augmentation randomly, harmlessly distorts each training image a little differently every time it's shown - flipping, rotating, zooming. This means the model effectively never sees the exact same image twice, forcing it to learn the *general* visual pattern of each flower instead of memorizing pixels.

- **`RandomFlip('horizontal')`** - randomly mirrors the image left-right. A flower photographed from the left still looks correct flipped.
- **`RandomRotation(0.1)`** - randomly rotates by up to 10% of a full turn (±36°), since flower photos aren't always taken perfectly upright.
- **`RandomZoom(0.1)`** - randomly zooms in/out by up to 10%, helping the model recognize flowers at different distances.

> **Note on the API:** the original notebook used `layers.experimental.preprocessing.RandomFlip` (etc.), which was the *experimental* location for these layers in older TensorFlow versions. They've since graduated to the stable `layers.RandomFlip` (etc.) namespace, which is what we use below.

We wrap the augmentation steps in their own small `Sequential` model, which we can then drop directly into the front of our real model (see Section 8) - Keras automatically applies it only during training and skips it during evaluation/prediction.

In [ ]:
data_augmentation = Sequential([
    layers.RandomFlip('horizontal',
                       input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='data_augmentation')

Let's visualize what augmentation actually does to one image -- several different random augmentations of the same photo:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

axes[0].imshow(X[0])
axes[0].set_title('Original')
axes[0].axis('off')

example_image = np.expand_dims(X[0], 0)
for i in range(1, 5):
    augmented = data_augmentation(example_image, training=True)
    axes[i].imshow(augmented[0].numpy().astype('uint8'))
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

---

## 8. Augmented Model (With Dropout)

We rebuild the same CNN architecture as the baseline, with two additions:

- **`data_augmentation` as the very first layer** - every training image passes through the random flip/rotate/zoom steps before reaching the convolutional layers.
- **`Dropout(0.2)`** - during training, this layer randomly "turns off" 20% of the values flowing through it on each pass. This prevents the network from becoming overly reliant on any single feature, which is another standard tool (alongside augmentation) for fighting overfitting.

Everything else - architecture, optimizer, loss, number of epochs - stays identical to the baseline, so the comparison between the two runs isolates the effect of these two changes.

In [ ]:
augmented_model = Sequential([
    data_augmentation,
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes),
])

augmented_model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

augmented_history = augmented_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=30,
    verbose=2,
)

In [ ]:
plot_history(augmented_history, 'Augmented + Dropout Model')

In [ ]:
augmented_test_loss, augmented_test_accuracy = augmented_model.evaluate(
    X_test_scaled, y_test, verbose=0
)
print(f'Augmented model test accuracy: {augmented_test_accuracy:.4f}')

---

## 9. Baseline vs. Augmented: Side-by-Side

The real story here isn't just the final test accuracy number - it's the **gap between training and validation accuracy** in the curves above. A smaller gap means the model is generalizing better rather than memorizing the training set.

In [ ]:
print(f'{"Model":<25}{"Test Accuracy":>15}')
print('-' * 40)
print(f'{"Baseline (no augmentation)":<25}{baseline_test_accuracy:>15.4f}')
print(f'{"Augmented + Dropout":<25}{augmented_test_accuracy:>15.4f}')

## Summary

- The **baseline model** trains without any protection against overfitting, so its training accuracy tends to climb well past its validation accuracy over 30 epochs - a clear overfitting signal.
- **Data augmentation** (random flip/rotate/zoom) prevents the model from ever seeing the exact same image twice, forcing it to learn general flower features instead of memorizing specific photos.
- **Dropout** independently reduces overfitting by preventing the network from relying too heavily on any single learned feature.
- Together, these typically produce a smaller train/validation gap and a validation/test accuracy that's more representative of how the model will perform on genuinely new flower photos.